# FIT5202 Data processing for Big data

##  Week 1 Lab: Getting started with Apache Spark

Welcome to your first FIT5202 lab! 🎉 This notebook introduces the environment and the basic Spark DataFrame operations that we will use throughout the semester.
> **Before you begin:** This notebook assumes that your Docker container is already installed and running. If the container is not working, return to the **Environment Setup Guide** or ask your tutor for help.

### What is Apache Spark?
**Apache Spark** is a fast and general engine for large-scale data processing. It has been reported that Spark is **100x faster** than Hadoop MapReduce in memory and **10x faster** on disk. Apache Spark is designed to write applications quickly in Java, Scala or Python. It aims to provide a big data processing framework that can be used for streaming data manipulation (Spark streaming), machine learing and batch processing (Hadoop integration). Spark SQL is a Spark module for structured data processing. One use of Spark SQL is to execute SQL queries. A Dataset is a distributed collection of data.

**Today's goal** is simple: confirm Spark is running in your container, touch it with your own hands, you do not need to understand Spark's full internal architecture today. The goal of this first lab is to become comfortable starting Spark and working with a DataFrame.

## Today's Plan

1. confirm that Python and PySpark are available;
2. configure and start a Spark application;
3. explain the roles of `SparkConf`, `SparkSession`, and `SparkContext`;
4. create and inspect a Spark DataFrame;
5. use `select()` and `filter()`;
6. combine basic DataFrame operations; and
7. compare a small Python solution with a Spark solution.


### Notebook Shortcuts
<font color='blue'>
    <strong>Notebook shortcuts you need to be familiar with and use frequently:</strong>
    
- Run your cells using SHIFT+ENTER (or "Run cell")
- Run the current cell and insert a new cell below: ALT+ENTER
- To see more commands, please click the "menu" option (e.g. "Insert", "Cell")
- To see more keyboard shortcuts, click the above "keyboard image" button. Use "Esc" to enter command mode. Then, you can use a command. Some of the popular shortcuts are 
    - Basic navigation: enter, shift-enter, up/k, down/j
    - Saving the notebook: s
    - Cell creation: `a` = insert cell above, `b` = insert cell below
</font>
    
Let's get started.

## Table of Contents

1. [Check the Environment](#part-1)
2. [Get Started with Spark](#part-2)
   - [Spark Configuration](#spark-configuration)
   - [SparkSession and SparkContext](#spark-session-context)
   - [Spark UI and Logs](#spark-ui)
3. [DataFrames in Spark](#part-3)
4. [Basic DataFrame Operations](#part-4)
5. [Python and Spark](#part-5)
6. [If You Finish Early](#extra-practice)
7. [Stop Spark](#stop-spark)

<a id="part-1"></a>
# Part 1 - Check the Environment

Your Docker container already comes with Python, PySpark and Jupyter installed. We're not installing anything here, just confirming it all actually works.

Run the cell below. You should see a Python version and a PySpark version, with no errors.

In [2]:
import sys
import pyspark

print("Python version:", sys.version.split()[0])
print("PySpark version:", pyspark.__version__)

Python version: 3.13.12
PySpark version: 4.1.1


<a id="part-2"></a>
# Part 2 - Get Started with Spark

<a id="spark-configuration"></a>
## Spark Configuration

Before running a Spark application, we need to specify some basic configuration settings.

`SparkConf` stores configuration information for a Spark application, such as:
- where Spark should run;
- how many processor cores it can use; and
- the application name shown in the Spark UI.

In this lab, Spark will run locally inside the Docker container.

In [4]:
# Import SparkConf
from pyspark import SparkConf

# Run Spark locally using all available logical processor cores.
# If we want Spark to run locally with 'k' worker threads, we can specify as "local[k]".
master = "local[*]"

# The `appName` field is a name to be shown on the Spark cluster UI page
app_name = "Introduction to Apache Spark"

# Create the Spark configuration.
spark_conf = (
    SparkConf()
    .setMaster(master)
    .setAppName(app_name)
)

### What does `local[*]` mean?

- `local` means that Spark runs on this computer rather than on a remote cluster.
- `[*]` means that Spark can use all available logical processor cores.
- For example, `local[2]` would allow Spark to use two worker threads.

<a id="spark-session-context"></a>
## SparkSession and SparkContext

Two important Spark objects are:

- **SparkSession** — the main entry point for working with DataFrames and Spark SQL;
- **SparkContext** — the connection to Spark's underlying execution environment.

Apache Spark community released a powerful Python package, **`PySpark`**. Using **`PySpark`**, we can  initialise Spark, create DataFrame from the data, sort, filter and sample the data. 

Especially, we will use and import **`SparkContext`** from **`pyspark`**, which is the main entry point for Spark Core functionality. The **`SparkSession`** object provides methods used to create DataFrames from various input sources.  

Spark applications run as independent sets of processes on a cluster, which is specified by the **`SparkContext`** object. **`SparkContext`** can connect to several types of cluster managers (local (standalone), Mesos or YARN), which allocate resources across applications. Once connected, Spark acquires executors on nodes in the cluster, which are processes that run computations and store data for your application. Next, it sends your application code (passed to `SparkContext`) to the executors. Finally, **`SparkContext`** sends tasks to the executors to run.

However, only one SparkContext instance should be active per JVM. There are two ways to create a SparkContext as shown in the next block of code. Since we will only use DataFrames for this tutorial, we will use the first method which creates the SparkContext via SparkSession.

**`PySpark`** applications start with initializing **`SparkSession`** which is the entry point of PySpark as below:

### Method 1 — Create Spark through `SparkSession`

In [5]:
# Import SparkContext and SparkSession
from pyspark import SparkContext
from pyspark.sql import SparkSession

# Create or retrieve a SparkSession using the configuration above.
spark = (
    SparkSession.builder
    .config(conf=spark_conf)
    .getOrCreate()
)

# Access the SparkContext associated with this SparkSession.
sc = spark.sparkContext

# Reduce unnecessary Spark log messages so notebook output is easier to read.
sc.setLogLevel("ERROR")

We will use:

- `spark` when creating and processing DataFrames;
- `sc` when we need access to lower-level Spark functionality.

Run the next cell to confirm that Spark is running with the expected configuration.

In [6]:
# Display the application name stored in the active SparkContext.
print("Application name:", sc.appName)

# Display the master setting used by the active SparkContext.
print("Master:", sc.master)

Application name: Introduction to Apache Spark
Master: local[*]


### Method 2 — Create or retrieve a `SparkContext` directly

A Spark application can also create or retrieve a `SparkContext` directly.

This method is more commonly associated with lower-level Spark Core and RDD operations.

Because this lab focuses on DataFrames, we will use **Method 1**.

In [5]:
# Alternative method: create or retrieve a SparkContext directly.
# This approach is more common when working with lower-level Spark Core or RDD operations.
#
# sc = SparkContext.getOrCreate(spark_conf)

# Reduce unnecessary Spark log messages if this alternative method is used.
# sc.setLogLevel("ERROR")

# Note:
# getOrCreate() returns the existing SparkContext if one is already active.
# Otherwise, it creates a new SparkContext using spark_conf.


## How These Objects Fit Together

A simple way to remember their roles is:

SparkConf -> stores application settings

SparkSession -> main entry point for DataFrames and Spark SQL

SparkContext -> connects the application to Spark's execution environment

<a id="spark-ui"></a>
## Open the Spark UI

While your `SparkSession` is alive, Spark runs a small web dashboard. Open a browser tab to: **http://localhost:4040**

The page may look mostly empty at first. That is normal. Later in the semester, we will use it to inspect Spark jobs, stages and tasks.
For today, simply confirm that the page opens.

*(If the page doesn't load, check that the port is mapped in your Docker setup: see the Docker Setup Guide.)*

## Know where logs and errors show up

During the semester, errors and diagnostic information may appear in three places:

1. **The cell output, right here in the notebook** — Python-level errors (typos, wrong arguments, etc.) and anything you `print()` or `show()` appear directly below the cell you ran.
2. **Your container's terminal / `docker compose logs`** — this is where the underlying Java/Spark process prints its own logs, warnings, and stack traces. If a cell just hangs or dies with no clear Python error, check here first — it's also where you'll notice things like the container running out of memory.
3. **The Spark UI (http://localhost:4040)** — once jobs have actually run, this is where you can inspect *how* Spark executed them (stages, tasks, timings). Not usually needed for basic errors, but essential later for performance debugging.

For basic coding errors, start with the notebook output. For container or Spark process problems, check the terminal logs.

### Checkpoint
Run the next cell. If it prints the Spark version without an error, your Spark environment is ready.

In [1]:
# Read the version number from the active SparkSession.
spark_version = spark.version

# Display a confirmation message and the Spark version.
print(f"Spark is running in my container! Spark version: {spark_version}")

NameError: name 'spark' is not defined

<a id="part-3"></a>
# Part 3 - DataFrames in Spark

A Spark DataFrame is a distributed collection of data organised into named columns.

It is similar to:

- a table in a relational database;
- a pandas or R DataFrame; or
- a worksheet containing structured rows and columns.

A DataFrame contains:

- **rows** — individual records;
- **columns** — named attributes; and
- a **schema** — the name and data type of each column.

Spark DataFrames are designed to work with large datasets and can distribute processing across available resources.

Useful references:

- [PySpark DataFrame quick start](https://spark.apache.org/docs/latest/api/python/getting_started/quickstart_df.html)
- [PySpark SQL API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html)


## Creating DataFrames 
Data can be loaded from <b>csv, json, xml</b> and other sources like <b>local file system</b> or <b>HDFS</b>. More information on : 
https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html

SparkSession provides an easy method <code>createDataFrame</code> to create Spark DataFrames. To display the schema, i.e. the  structure of the DataFrame, you can use <strong>printSchema()</strong> method.

For this first example, we will create a small DataFrame directly in Python so that we can focus on the syntax.

The `SparkSession.createDataFrame()` method converts a collection of Python records into a Spark DataFrame.

In [33]:
# Create a Python list containing six records.
# Each tuple represents one person.
people_data = [
    ("Alice", 34, "Melbourne", "Data Analyst"),
    ("Bob", 45, "Sydney", "Engineer"),
    ("Charlie", 25, "Melbourne", "Student"),
    ("Dana", 31, "Brisbane", "Researcher"),
    ("Ethan", 29, "Sydney", "Developer"),
    ("Fatima", 38, "Melbourne", "Lecturer")
]

# Define the column name associated with each value in the tuples above.
column_names = ["name", "age", "city", "occupation"]

# Convert the Python records into a Spark DataFrame.
people_df = spark.createDataFrame(people_data, column_names)


In [8]:
# Display the first 20 rows of people_df using show()'s default behaviour.
people_df.show()

# Display only the first three rows.
people_df.show(3)

# Display the schema, including column names and inferred data types.
people_df.printSchema()

# Read the list of column names and print it with a clear label.
print("Column names:", people_df.columns)

# Trigger a Spark action that counts all rows, then print the result.
print("Number of rows:", people_df.count())


+-------+---+---------+------------+
|   name|age|     city|  occupation|
+-------+---+---------+------------+
|  Alice| 34|Melbourne|Data Analyst|
|    Bob| 45|   Sydney|    Engineer|
|Charlie| 25|Melbourne|     Student|
|   Dana| 31| Brisbane|  Researcher|
|  Ethan| 29|   Sydney|   Developer|
| Fatima| 38|Melbourne|    Lecturer|
+-------+---+---------+------------+

+-------+---+---------+------------+
|   name|age|     city|  occupation|
+-------+---+---------+------------+
|  Alice| 34|Melbourne|Data Analyst|
|    Bob| 45|   Sydney|    Engineer|
|Charlie| 25|Melbourne|     Student|
+-------+---+---------+------------+
only showing top 3 rows
root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- city: string (nullable = true)
 |-- occupation: string (nullable = true)

Column names: ['name', 'age', 'city', 'occupation']
Number of rows: 6


### A useful habit: meet your data

Whenever you receive a new DataFrame, begin with these four checks:

```python
df.show(5)
df.printSchema()
df.columns
df.count()
```

Together, they answer:

1. What does the data look like?
2. Which columns exist?
3. What type of data is stored in each column?
4. How many rows are there?

#### Lab Task 1
<a class="anchor" id="lab-task-1"></a>

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">1. Lab Task: </strong> Without changing <b>people_df</b>, complete the code below to:

<ol>
    <li>display only the first four rows;</li>
    <li>print its schema;</li>
    <li>display its list of column names; and</li>
    <li>count its rows.</li>
</ol>

<strong>COMPLETE THE CODE BELOW.</strong>
</div>

In [34]:
# Lab Task 1

# 1. Display only the first four rows
# YOUR ANSWER HERE
people_df.show(4)

# 2. Print the schema
# YOUR ANSWER HERE
people_df.printSchema()

# 3. Display the list of column names
# YOUR ANSWER HERE
print("Column names:", people_df.columns)

# 4. Count the number of rows
# YOUR ANSWER HERE
print("Number of rows:", people_df.count())


+-------+---+---------+------------+
|   name|age|     city|  occupation|
+-------+---+---------+------------+
|  Alice| 34|Melbourne|Data Analyst|
|    Bob| 45|   Sydney|    Engineer|
|Charlie| 25|Melbourne|     Student|
|   Dana| 31| Brisbane|  Researcher|
+-------+---+---------+------------+
only showing top 4 rows
root
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- city: string (nullable = true)
 |-- occupation: string (nullable = true)

Column names: ['name', 'age', 'city', 'occupation']
Number of rows: 6


<a id="part-4"></a>
# Part 4 - Basic DataFrame Operations <a class="anchor" id="section3_df_select"></a>

Today we will use only two operations for changing what we see:
- `select()` chooses columns;
- `filter()` chooses rows.

These operations can also be chained together. We will study how Spark executes them in more detail later.

In [35]:
# Select and display only the name column.
people_df.select("name").show()

# Select and display the name and occupation columns.
people_df.select("name", "occupation").show()

# Keep rows where age is greater than 30, then display the result.
people_df.filter(people_df["age"] > 30).show()

# Keep rows where city is Melbourne, then display the result.
people_df.filter(people_df["city"] == "Melbourne").show()

# Start with people_df.
(
    people_df

    # Keep only rows for people who live in Melbourne.
    .filter(people_df["city"] == "Melbourne")

    # Keep only the name and occupation columns.
    .select("name", "occupation")

    # Display the final result.
    .show()
)

+-------+
|   name|
+-------+
|  Alice|
|    Bob|
|Charlie|
|   Dana|
|  Ethan|
| Fatima|
+-------+

+-------+------------+
|   name|  occupation|
+-------+------------+
|  Alice|Data Analyst|
|    Bob|    Engineer|
|Charlie|     Student|
|   Dana|  Researcher|
|  Ethan|   Developer|
| Fatima|    Lecturer|
+-------+------------+

+------+---+---------+------------+
|  name|age|     city|  occupation|
+------+---+---------+------------+
| Alice| 34|Melbourne|Data Analyst|
|   Bob| 45|   Sydney|    Engineer|
|  Dana| 31| Brisbane|  Researcher|
|Fatima| 38|Melbourne|    Lecturer|
+------+---+---------+------------+

+-------+---+---------+------------+
|   name|age|     city|  occupation|
+-------+---+---------+------------+
|  Alice| 34|Melbourne|Data Analyst|
|Charlie| 25|Melbourne|     Student|
| Fatima| 38|Melbourne|    Lecturer|
+-------+---+---------+------------+

+-------+------------+
|   name|  occupation|
+-------+------------+
|  Alice|Data Analyst|
|Charlie|     Student|
| Fa

#### Lab Task 2
<a class="anchor" id="lab-task-2"></a>

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">2. Lab Task: </strong> Using <b>people_df</b>, complete the code below to:

<ol>
    <li>display the <b>name</b>, <b>age</b> and <b>city</b> columns;</li>
    <li>display only people who are younger than 35; and</li>
    <li>display the <b>name</b> and <b>age</b> of people older than 30 by combining <b>filter()</b> and <b>select()</b>.</li>
</ol>

<strong>COMPLETE THE CODE BELOW.</strong>
</div>

In [36]:
# Lab Task 2

# 1. Display the name, age and city columns
# YOUR ANSWER HERE
people_df.select("name", "age", "city").show()

# 2. Display only people who are younger than 35
# YOUR ANSWER HERE
people_df.filter(people_df["age"] < 35).show()

# 3. Display the name and age of people older than 30
# Filter first, then select.
# YOUR ANSWER HERE
(
    people_df

    .filter(people_df["age"] > 30)

    .select("name", "age")

    .show()
)

+-------+---+---------+
|   name|age|     city|
+-------+---+---------+
|  Alice| 34|Melbourne|
|    Bob| 45|   Sydney|
|Charlie| 25|Melbourne|
|   Dana| 31| Brisbane|
|  Ethan| 29|   Sydney|
| Fatima| 38|Melbourne|
+-------+---+---------+

+-------+---+---------+------------+
|   name|age|     city|  occupation|
+-------+---+---------+------------+
|  Alice| 34|Melbourne|Data Analyst|
|Charlie| 25|Melbourne|     Student|
|   Dana| 31| Brisbane|  Researcher|
|  Ethan| 29|   Sydney|   Developer|
+-------+---+---------+------------+

+------+---+
|  name|age|
+------+---+
| Alice| 34|
|   Bob| 45|
|  Dana| 31|
|Fatima| 38|
+------+---+



#### Lab Task 3
<a class="anchor" id="lab-task-3"></a>

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
<strong style="color:#FF5555">3. Lab Task: </strong> The <b>sales_df</b> DataFrame contains product sales data. Apply the DataFrame operations learned in Part 4 to:

<ol>
    <li>display the first five rows, schema, column names and row count;</li>
    <li>display only the <b>product</b>, <b>price</b> and <b>quantity</b> columns;</li>
    <li>display products with a price greater than 100;</li>
    <li>display products in the <b>Electronics</b> category;</li>
    <li>display only the <b>product</b> and <b>price</b> of products with a price greater than 100; and</li>
    <li>write and answer one simple business question using <b>select()</b> and/or <b>filter()</b>.</li>
</ol>

<strong>COMPLETE THE CODE BELOW.</strong>
</div>

In [37]:
# Lab Task 3

# Create a Python list containing product sales records.
# Each tuple stores: product_id, product, category, price, and quantity.
sales_data = [
    (1001, "Laptop", "Electronics", 1200.0, 2),
    (1002, "Mouse", "Electronics", 35.0, 8),
    (1003, "Desk", "Furniture", 450.0, 3),
    (1004, "Chair", "Furniture", 220.0, 5),
    (1005, "Monitor", "Electronics", 380.0, 4),
    (1006, "Notebook", "Stationery", 6.5, 20),
    (1007, "Pen Set", "Stationery", 12.0, 15),
    (1008, "Keyboard", "Electronics", 95.0, 6)
]

# Define the column names in the same order as the tuple values above.
sales_columns = [
    "product_id",
    "product",
    "category",
    "price",
    "quantity"
]

# Convert the Python sales records into a Spark DataFrame.
sales_df = spark.createDataFrame(sales_data, sales_columns)

# 1. Meet the data using show(), printSchema(), columns, and count().
# YOUR ANSWER HERE
sales_df.show()
sales_df.printSchema()
sales_df.columns
sales_df.count()

# 2. Select product, price, and quantity.
# YOUR ANSWER HERE
sales_df.select("product", "price", "quantity").show()

# 3. Display products with a price greater than 100.
# YOUR ANSWER HERE
sales_df.filter(sales_df["price"] < 100).show()

# 4. Display products in the Electronics category.
# YOUR ANSWER HERE
sales_df.filter(sales_df["category"] == "Electronics").show()

# 5. Filter products over 100, then select product and price.
# YOUR ANSWER HERE
(
    sales_df

    .filter(sales_df["price"] > 100)

    .select("product", "price")

    .show()
)

# 6. Write one simple business question.
# My question:


# Answer your question using select() and/or filter().
# YOUR ANSWER HERE


+----------+--------+-----------+------+--------+
|product_id| product|   category| price|quantity|
+----------+--------+-----------+------+--------+
|      1001|  Laptop|Electronics|1200.0|       2|
|      1002|   Mouse|Electronics|  35.0|       8|
|      1003|    Desk|  Furniture| 450.0|       3|
|      1004|   Chair|  Furniture| 220.0|       5|
|      1005| Monitor|Electronics| 380.0|       4|
|      1006|Notebook| Stationery|   6.5|      20|
|      1007| Pen Set| Stationery|  12.0|      15|
|      1008|Keyboard|Electronics|  95.0|       6|
+----------+--------+-----------+------+--------+

root
 |-- product_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: long (nullable = true)

+--------+------+--------+
| product| price|quantity|
+--------+------+--------+
|  Laptop|1200.0|       2|
|   Mouse|  35.0|       8|
|    Desk| 450.0|       3|
|   Chair| 220.0|       5|
| Monitor|

<a id="part-5"></a>
# Part 5 — Python and Spark: Similar Goal, Different Scale

You may notice that a Spark DataFrame operation can look similar to ordinary Python code.

For a tiny dataset, normal Python is usually simpler. Spark becomes useful when the data is too large for one machine or when the work needs to be distributed across multiple machines.

For now, compare the *style* of the two solutions. We will explain how Spark processes data behind the scenes in a later lecture.


## Filter data using normal Python

The following Python code keeps people older than 30.


## Filter data using normal Python

The following Python code keeps people older than 30.


## Filter data using normal Python

The following Python code keeps people older than 30.


#### 

In [38]:
# Create a small Python list containing names and ages.
people = [
    ("Alice", 34),
    ("Bob", 45),
    ("Charlie", 25),
    ("Dana", 31)
]

# Create an empty list to store records that satisfy the condition.
adults_over_30 = []

# Read one name and age from the people list at a time.
for name, age in people:
    if age > 30:
        adults_over_30.append((name, age))

print(adults_over_30)

[('Alice', 34), ('Bob', 45), ('Dana', 31)]


## Filter data using a Spark DataFrame

The Spark version expresses the same goal using `filter()`.

In [39]:
# Convert the same Python records into a Spark DataFrame.
comparison_df = spark.createDataFrame(
    people,
    ["name", "age"]
)

# Keep rows where age is greater than 30, then display the result.
comparison_df.filter(comparison_df["age"] > 30).show()

+-----+---+
| name|age|
+-----+---+
|Alice| 34|
|  Bob| 45|
| Dana| 31|
+-----+---+



### Think about it
Both examples keep records where `age > 30`.

- Which version is easier to read for this tiny dataset?
- Why might Spark still be useful for a dataset containing millions or billions of rows?

You do not need to understand Spark's internal execution yet. The following weeks we will explain why the Spark version can scale beyond one machine.

<a id="extra-practice"></a>
# If You Finish Early

Use <code>resources/bank.csv</code> to practise applying the Week 1 DataFrame operations more independently.

<div style="background:#fff3cd;padding:10px;border-radius:4px">
<strong>Important:</strong> Complete the notebook from the beginning before attempting the extra practice activities. Your <code>spark</code> session and earlier DataFrames must already be available.
</div>

#### Extra Practice 1
<a class="anchor" id="Extra-Practice-1"></a>

<div style="background:rgba(0,109,174,0.12);padding:10px;border-radius:4px">
<strong style="color:#FF5555">Extra Practice 1: Explore a new dataset</strong>

<p>Load <b>resources/bank.csv</b> using <b>header=True</b> and <b>inferSchema=True</b>.</p>

<p>Then:</p>

<ol>
    <li>display the first five rows;</li>
    <li>print the schema;</li>
    <li>display the column names;</li>
    <li>count the rows;</li>
    <li>select two or three useful columns; and</li>
    <li>filter the data using one numeric condition.</li>
</ol>
</div>

In [40]:
# Extra Practice 1

# Read bank.csv as a Spark DataFrame.
bank_df = spark.read.csv(
    "resources/bank.csv",
    # Use the first line of the file as the column names.
    header=True,
    # Ask Spark to infer suitable data types instead of treating every column as text.
    inferSchema=True
)

# 1. Display the first five rows
# YOUR ANSWER HERE
bank_df.show(5)

# 2. Print the schema
# YOUR ANSWER HERE
bank_df.printSchema()

# 3. Display the column names
# YOUR ANSWER HERE
bank_df.columns

# 4. Count the rows
# YOUR ANSWER HERE
bank_df.count()

# 5. Select two or three useful columns
# YOUR ANSWER HERE
bank_df.select("age", "job", "balance").show()

# 6. Filter using one numeric condition
# YOUR ANSWER HERE
bank_df.filter(bank_df["balance"] > 50000).show()

+---+----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
|age|       job|marital|education|default|balance|housing|loan|contact|day|month|duration|campaign|pdays|previous|poutcome|deposit|
+---+----------+-------+---------+-------+-------+-------+----+-------+---+-----+--------+--------+-----+--------+--------+-------+
| 59|    admin.|married|secondary|     no|   2343|    yes|  no|unknown|  5|  may|    1042|       1|   -1|       0| unknown|    yes|
| 56|    admin.|married|secondary|     no|     45|     no|  no|unknown|  5|  may|    1467|       1|   -1|       0| unknown|    yes|
| 41|technician|married|secondary|     no|   1270|    yes|  no|unknown|  5|  may|    1389|       1|   -1|       0| unknown|    yes|
| 55|  services|married|secondary|     no|   2476|    yes|  no|unknown|  5|  may|     579|       1|   -1|       0| unknown|    yes|
| 54|    admin.|married| tertiary|     no|    184|     no|  no|unknown|  5| 

#### Extra Practice 2
<a class="anchor" id="Extra-Practice-2"></a>

<div style="background:rgba(0,109,174,0.12);padding:10px;border-radius:4px">
<strong style="color:#FF5555">Extra Practice 2: Change the operation order</strong>

<p>Try both of the following patterns:</p>

<pre><code>bank_df.filter(...).select(...)</code></pre>

<pre><code>bank_df.select(...).filter(...)</code></pre>

<p>Do both versions work? What happens if the column required by <b>filter()</b> is not included in <b>select()</b>?</p>
</div>

In [41]:
# Extra Practice 2

# A. Filter first, then select.
# Choose a column for the filter condition.
# After filtering, select two or three columns to display.
# YOUR ANSWER HERE
bank_df.filter(bank_df["balance"] > 50000).select("age", "balance", "duration").show()

# B. Select first, then filter.
# First select a set of columns that still includes the column needed by filter().
# Then apply the same filtering condition.
# YOUR ANSWER HERE
bank_df.select("age", "balance","duration").filter(bank_df["balance"] > 50000).show()

# Compare the two results.
# What happens if select() removes the column required by filter()?
bank_df.select("age", "duration").filter(bank_df["balance"] > 50000).show()

# Write your observation below.
# YOUR ANSWER HERE

+---+-------+--------+
|age|balance|duration|
+---+-------+--------+
| 61|  52587|     290|
| 84|  81204|     679|
| 61|  52587|     394|
| 84|  81204|     390|
| 52|  66653|     109|
| 43|  56831|     243|
| 56|  51439|     325|
+---+-------+--------+

+---+-------+--------+
|age|balance|duration|
+---+-------+--------+
| 61|  52587|     290|
| 84|  81204|     679|
| 61|  52587|     394|
| 84|  81204|     390|
| 52|  66653|     109|
| 43|  56831|     243|
| 56|  51439|     325|
+---+-------+--------+

+---+--------+
|age|duration|
+---+--------+
| 61|     290|
| 84|     679|
| 61|     394|
| 84|     390|
| 52|     109|
| 43|     243|
| 56|     325|
+---+--------+



#### Extra Practice 3
<a class="anchor" id="Extra-Practice-3"></a>

<div style="background:rgba(0,109,174,0.12);padding:10px;border-radius:4px">
<strong style="color:#FF5555">Extra Practice 3: Create a new column and ask your own question</strong>

<ol>
    <li>Use <b>withColumn()</b> to create one new column.</li>
    <li>Write one simple question that can be answered using <b>select()</b>, <b>filter()</b>, or <b>withColumn()</b>.</li>
    <li>Write Spark code to answer your question.</li>
</ol>
</div>

In [42]:
# Import col(), which lets us refer to a DataFrame column inside an expression.
from pyspark.sql.functions import col

# Start with the existing people_df DataFrame.
people_df.withColumn(
    # Name the new column age_next_year.
    "age_next_year",

    # Read each value from the existing age column and add 1.
    col("age") + 1

# Display the resulting DataFrame.
).show()


+-------+---+---------+------------+-------------+
|   name|age|     city|  occupation|age_next_year|
+-------+---+---------+------------+-------------+
|  Alice| 34|Melbourne|Data Analyst|           35|
|    Bob| 45|   Sydney|    Engineer|           46|
|Charlie| 25|Melbourne|     Student|           26|
|   Dana| 31| Brisbane|  Researcher|           32|
|  Ethan| 29|   Sydney|   Developer|           30|
| Fatima| 38|Melbourne|    Lecturer|           39|
+-------+---+---------+------------+-------------+



In [ ]:
# Extra practice 3

from pyspark.sql.functions import col
# 1. Create one new column in bank_df.
# Example pattern:
# bank_df.withColumn(
#     "new_column_name",
#     col("existing_numeric_column") * 2
# ).show()
#
# Replace both example column names with suitable names from bank_df.
# YOUR ANSWER HERE
bank_df.withColumn(
    "new_column_name",
    col("existing_numeric_column") * 2
).show()

# 2. Write one simple question about the bank dataset.
# The question should be answerable using select(), filter(), or withColumn().
# My question:


# 3. Write Spark code that answers your question.
# YOUR ANSWER HERE

## Stop Spark

Run the next cell only when you have completely finished the notebook.

Stopping Spark releases the resources used by the active `SparkSession`.

In [ ]:
spark.stop()
print(
    "SparkSession stopped. Congratulations on completing your first "
    "FIT5202 Spark lab! See you next week."
)